# Self-learning homophonic carver — Colab T4 (continual, disconnect-proof, monitored)
Runs CONTINUALLY (auto-restarts on any error), checkpoints to Google Drive (auto-resumes after a disconnect), keeps the runtime awake, scores each round with the Nemotron judge, and pushes `status.json` to a `selflearn-status` branch for remote monitoring.
Runtime → Change runtime type → **T4 GPU**, then Run all (the training cell loops until you stop it).

In [2]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-4236056b-2b71-eb4e-40bc-03499cabbd76)


In [3]:
# --- Drive: checkpoints + status survive Colab disconnects ---
from google.colab import drive
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/homophonic-carver'
import os; os.makedirs(CKPT, exist_ok=True)

Mounted at /content/drive


In [7]:
# --- secrets (Colab: 🔑 left sidebar -> add as named secrets) ---
from google.colab import userdata
def secret(name):
    try: return userdata.get(name)
    except Exception: return ''
os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY')  # LLM judge
os.environ['GITHUB_TOKEN']      = secret('GITHUB_TOKEN')         # status push + private clone
os.environ['GITHUB_REPO']       = 'grundlagen/Lingua-Sound-Wave'

In [6]:
# --- clone the repo (token used only if private) ---
BRANCH = 'claude/phrase-weave-multiword'
tok = os.environ.get('GITHUB_TOKEN','')
url = f"https://{tok+'@' if tok else ''}github.com/{os.environ['GITHUB_REPO']}.git"
!rm -rf repo && git clone --depth 1 -b $BRANCH $url repo
%cd repo/research/homophone-bench

Cloning into 'repo'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'repo/research/homophone-bench'
/content


In [ ]:
!apt-get -qq install -y espeak-ng >/dev/null
!pip -q install transformers trl datasets accelerate panphon wordfreq numpy
!python selflearn/reward.py   # sanity: prosody reward runs

In [ ]:
# --- keep-alive: reduce idle disconnects (keep the tab open) ---
from IPython.display import Javascript, display
display(Javascript('''function _k(){try{document.querySelector('colab-connect-button').shadowRoot.querySelector('#connect').click();}catch(e){}} setInterval(_k,60000);'''))
print('keep-alive armed')

In [ ]:
# --- CONTINUAL self-learning: auto-restarts on ANY error, never press Run again ---
# loops forever, checkpoints each round to Drive, pushes status; stop the cell when satisfied.
%cd selflearn
!python run_continual.py --base Qwen/Qwen2.5-1.5B-Instruct \
    --k 8 --keep_thresh 0.55 --ckpt_dir "$CKPT" --eval_llm

In [ ]:
# --- try the self-learned carver (run anytime; loads from the Drive checkpoint) ---
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
t = AutoTokenizer.from_pretrained(CKPT)
m = AutoModelForCausalLM.from_pretrained(CKPT, torch_dtype=torch.float16, device_map='auto')
for en in ['the little cat', 'my friend loves the sea', 'gentle rain falls']:
    p = f'Rewrite the English so it sounds the same when read aloud in French: {en}'
    ids = t.apply_chat_template([{'role':'user','content':p}], add_generation_prompt=True, return_tensors='pt').to(m.device)
    o = m.generate(ids, max_new_tokens=24, do_sample=False)
    print(en, '->', t.decode(o[0][ids.shape[1]:], skip_special_tokens=True).strip())

## Continual / monitor / admin
- **Continual + auto-restart:** the training cell runs `run_continual.py` — it loops rounds forever, skips a bad round, and relaunches (resuming from the Drive checkpoint) if the process dies. No re-pressing Run.
- **Resume:** if the runtime drops entirely, re-run the cells; it continues from the last checkpoint.
- **Monitor:** with `GITHUB_TOKEN`, each round pushes `selflearn/status.json` to the `selflearn-status` branch (round, kept, mean reward, sample carves + Nemotron scores). Ask Claude to read it and tune.
- **Trawl + experiment:** `python idea_trawler.py` catalogues ideas across all branches (`IDEAS.md`); `python experiment.py` sweeps parameters read-only (`experiments/results.jsonl`) — neither edits source.